In [1]:
import os
import sys
sys.path.append('..')
import torch
import librosa
from scipy.io import wavfile
from IPython import display as disp
from pathlib import Path
from tqdm import tqdm

from src.modules import MimiResynthesisModule, MimiResynthesisWithLogitsModule, MimiKMResynthesisModule, MimiResidualKMResynthesisModule, SpeakerModule
from anonymization.anonymizer import DemoAnonymizer

/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
/data/mtseng/official_repo/DarkStream/notebooks/../anonymization/demo_speaker_embeddings.py:5: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'spee

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "3" # Set the GPU device
MAX_WAV_VALUE = 32768.0

### Note
Use mimiresynthesis_nola for 0ms lookahead, mimiresynthesis_la7 for 140ms lookahead, and mimiresynthesis_la14 for 280ms lookahead model

MimiResynthesisModule -> for wav + CL archiecture

MimiKMResynthesisModule -> for wav + CL + KM architecture

MimiResynthesisWithLogitsModule -> For DarkStream with logits

MimiResidualKMResynthesisModule -> For DarkStream with residual KMeams

In [ ]:
speaker_encoder = SpeakerModule(device="cuda")

model_type = "darkstream_with_logits"  # Options: darkstream_without_kmeans, darkstream_with_logits, darkstream, darkstream_with_residualkmeans

resynthesizer = None
use_projector = False
if model_type == "darkstream_without_kmeans":
    resynthesizer = MimiResynthesisModule.load_from_checkpoint(
        "../ckpts/mimiresynthesis_la14.ckpt",
        encoder_path=None,
        map_location="cuda",
        decoder_pretrained=False
    )
elif model_type == "darkstream_with_logits":
    resynthesizer = MimiResynthesisWithLogitsModule.load_from_checkpoint(
        "../ckpts/mimiresynthesiswithlogits_la7.ckpt",
        encoder_path=None,
        map_location="cuda",
        decoder_pretrained=False
    )
    use_projector = True
elif model_type == "darkstream":
    resynthesizer = MimiKMResynthesisModule.load_from_checkpoint(
        "../ckpts/mimikmresynthesis_la7.ckpt",
        encoder_path=None,
        kmeans_path=None,
        map_location="cuda",
        decoder_pretrained=False
    )
elif model_type == "darkstream_with_residualkmeans":
    resynthesizer = MimiResidualKMResynthesisModule.load_from_checkpoint(
        "../ckpts/mimiresidualkmresynthesis_la7_nq4.ckpt",  # Supports nq2, nq4, nq8. ajust paths accordingly.
        encoder_path=None,
        kmeans_path="../ckpts/km256_centers.pth",
        map_location="cuda",
        decoder_pretrained=False
    )
else:
    raise ValueError(f"Unknown model type: {model_type}")
    
resynthesizer.eval()

/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


MimiResynthesisWithLogitsModule(
  (encoder): MimiPasrModule(
    (net): MimiEncoderModel(
      (encoder): SEANetEncoder(
        (model): Sequential(
          (0): StreamingConv1d(
            (conv): NormConv1d(
              (conv): RawStreamingConv1d(1, 64, kernel_size=(7,), stride=(1,))
            )
          )
          (1): SEANetResnetBlock(
            (block): Sequential(
              (0): ELU(alpha=1.0)
              (1): StreamingConv1d(
                (conv): NormConv1d(
                  (conv): RawStreamingConv1d(64, 32, kernel_size=(3,), stride=(1,))
                )
              )
              (2): ELU(alpha=1.0)
              (3): StreamingConv1d(
                (conv): NormConv1d(
                  (conv): RawStreamingConv1d(32, 64, kernel_size=(1,), stride=(1,))
                )
              )
            )
            (add): StreamingAdd()
            (shortcut): Identity()
          )
          (2): ELU(alpha=1.0)
          (3): StreamingConv1d(
       

In [4]:
wav_file = "../sample_wav/BDL/arctic_a0001.wav"
_device = "cuda"
sample_rate = 16000

audio, _ = librosa.load(wav_file, sr=sample_rate)

disp.Audio(audio, rate=16000)

### Non streaming synthesis

In [5]:
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(_device)
embed = speaker_encoder(audio.squeeze(0))
resyth_audio = resynthesizer(audio, embed)
resyth_audio = resyth_audio.squeeze()
# resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.detach().cpu().numpy() #.astype('int16')

disp.Audio(resyth_audio, rate=16000)

### VC


In [6]:
tgt_wav_file = "../sample_wav/SLT/arctic_a0001.wav"
tgtaudio, _ = librosa.load(tgt_wav_file, sr=sample_rate)
disp.Audio(tgtaudio, rate=16000)

In [7]:
tgtaudio = torch.from_numpy(tgtaudio).unsqueeze(0).unsqueeze(0).to(_device)

embed = speaker_encoder(tgtaudio.squeeze(0))
vc_audio = resynthesizer(audio, embed)
vc_audio = vc_audio.squeeze()
# vc_audio = vc_audio * MAX_WAV_VALUE
vc_audio = vc_audio.detach().cpu().numpy() #.astype('int16')

disp.Audio(vc_audio, rate=16000)

### Streaming synthesis (One way)

In [8]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    frame_size = 320 # 20ms for 16000Hz sample rate, incraease to increase chunk size

    # pad the end of audio so lookahead doesn't drop the last few frames
    lookahead_size = 14
    pad = torch.zeros((1, 1, lookahead_size * frame_size)).to(audio.device)
    audio_padded = torch.cat([audio, pad], dim=-1).to(audio.device)

    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):  # in steps of frame_size, so chunk size is 20ms
                frame = audio_padded[:, :, offset:offset + frame_size]
                z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                if use_projector:
                    z_embed = resynthesizer.encoder.code_predictor(z_embed)
                    z_embed = resynthesizer.projector(z_embed.transpose(1, 2)).transpose(1, 2)
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:02<00:00, 60.34it/s]


### Streaming synthesis (Another way)

In [9]:
resynthesizer.encoder.net.streaming_forever(batch_size=1)
resynthesizer.decoder.streaming_forever(batch_size=1)
with torch.no_grad():
    original_length = audio.shape[-1]
    frame_size = 320

    all_audios = []
    all_z_embeds = []

    for offset in tqdm(range(0, original_length, frame_size)):
        frame = audio[:, :, offset:offset + frame_size]
        z_embed = resynthesizer.encoder.net(frame)
        if z_embed.shape[-1] == 0:
            continue
        if use_projector:
            z_embed = resynthesizer.encoder.code_predictor(z_embed)
            z_embed = resynthesizer.projector(z_embed.transpose(1, 2)).transpose(1, 2)
        resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
        all_audios.append(resynth_audio_chunk)   
        all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:02<00:00, 68.26it/s]


### Anonymization
For anonymization use embedding from anonymizer model instead

In [10]:
anonymizer = DemoAnonymizer(
    model_path=Path('../anonymization/models'), 
    model_tag='gan',
    device="cuda"
)

Model type of anonymizer: gan


In [11]:
anon_embed = anonymizer.anonymize_embedding(audio.squeeze().cpu().numpy(), 16000)
anon_embed = torch.cat([anon_embed[192:], anon_embed[:192]], dim=0) #embeds are concatenated in the reversed order

In [12]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):
                frame = audio[:, :, offset:offset + frame_size]
                z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                if use_projector:
                    z_embed = resynthesizer.encoder.code_predictor(z_embed)
                    z_embed = resynthesizer.projector(z_embed.transpose(1, 2)).transpose(1, 2)
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, anon_embed.unsqueeze(0))
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:02<00:00, 68.39it/s]
